# Medical Model Selection Benchmark

Benchmark open-weight ~1B parameter language models on medical QA datasets to select the best base model for fine-tuning, before also adding our security and uncertainty heads.

The winning model is the one with the highest average accuracy across medical benchmarks.

In [ ]:
# This notebook is more of a standalone evaluator than a project integration which is why I've opted
# to keep the env variables and dependencies here instead of in pixi. Please tell me off and correct
# if this is less preferable and it can be nicely formatted into some scripts instead.

!pip install lm-eval torch transformers accelerate pandas python-dotenv

import importlib
import os

import lm_eval
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Need HF token for gemma, llama.
if not os.environ.get("HF_TOKEN"):
    print("Enter your token for the hub now while you have the chance")

MODELS = [
    {"id": "meta-llama/Llama-3.2-1B", "name": "Llama-3.2-1B"},
    {"id": "Qwen/Qwen2.5-1.5B", "name": "Qwen2.5-1.5B"},
    {"id": "HuggingFaceTB/SmolLM2-1.7B", "name": "SmolLM2-1.7B"},
    {"id": "stabilityai/stablelm-2-1_6b", "name": "StableLM-2-1.6B"},
    {"id": "allenai/OLMo-1B-hf", "name": "OLMo-1B"},
    {"id": "allenai/OLMo-2-0425-1B", "name": "OLMo-2-1B"},
    {"id": "google/gemma-3-1b-pt", "name": "Gemma-3-1B"},
]

TASKS = [
    "medqa_4options",
    "pubmedqa",
    "mmlu_clinical_knowledge",
    "mmlu_medical_genetics",
    "mmlu_anatomy",
    "mmlu_professional_medicine",
]

EVAL_SETTINGS = {
    "num_fewshot": 5,
    "batch_size": "auto",
    "dtype": "float16",
}

In [ ]:
# main model evaluation

results = {}
failures = {}

for index, model_info in enumerate(MODELS, 1):
    name = model_info["name"]
    model_id = model_info["id"]

    print(f"[{index}/{len(MODELS)}] Evaluating {name} ({model_id})...")

    try:
        result = lm_eval.simple_evaluate(
            model="hf",
            model_args=f"pretrained={model_id},dtype={EVAL_SETTINGS['dtype']}",
            tasks=TASKS,
            num_fewshot=EVAL_SETTINGS["num_fewshot"],
            batch_size=EVAL_SETTINGS["batch_size"],
            random_seed=42,
            numpy_random_seed=42,
            torch_random_seed=42,
        )

        results[name] = result
        print(f"  -> Done.")

    except Exception as e:
        failures[name] = str(e)
        print(f"  -> FAILED: {e}")

print("\n" + "=" * 60)
print(f"Completed: {len(results)}/{len(MODELS)} models")
if failures:
    print(f"\nFailures ({len(failures)}):")
    for name, err in failures.items():
        print(f"  - {name}: {err}")

In [ ]:
# results comparison

ACC_KEYS = ["acc,none", "acc_norm,none", "acc", "acc_norm"]

rows = []
for model_name, result in results.items():
    task_results = result.get("results", {})
    row = {"Model": model_name}
    accs = []
    for task in TASKS:
        task_data = task_results.get(task, {})
        acc = next((task_data[k] for k in ACC_KEYS if task_data.get(k) is not None), None)
        if acc is not None:
            acc = float(acc)
            row[task] = round(acc * 100, 2)
            accs.append(acc)
        else:
            row[task] = None
    row["Avg Accuracy (%)"] = round(sum(accs) / len(accs) * 100, 2) if accs else None
    rows.append(row)

df = pd.DataFrame(rows).sort_values("Avg Accuracy (%)", ascending=False).reset_index(drop=True)

print("=" * 80)
print("MEDICAL BENCHMARK COMPARISON — ALL MODELS")
print("=" * 80)
display(df)

if not df.empty and df["Avg Accuracy (%)"].notna().any():
    best = df.iloc[0]
    print(f"\nRecommendation: {best['Model']} — highest avg accuracy at {best['Avg Accuracy (%)']}%")